# Bài 25 — chuỗi luyện Easy · LeetCode #121 — Best Time to Buy and Sell Stock

**Mức độ:** Easy · **Chủ đề:** Array · Greedy · One Pass · **Python:** 3  
**Đề gốc:** [https://leetcode.com/problems/best-time-to-buy-and-sell-stock/](https://leetcode.com/problems/best-time-to-buy-and-sell-stock/)

> Mục tiêu: hiểu cách tự suy luận từ ví dụ → viết code → trace trạng thái → chứng minh đúng → tự làm bài biến thể. Đọc Markdown, dự đoán rồi mới chạy cell code tiếp theo.

**Lưu ý:** đây là notebook dành cho người mới. Các cell TODO chỉ định nghĩa hàm, không được tự gọi khi chưa giải.

**Lưu ý học:** các ô TODO không được tự chạy; hãy tự làm, mở gợi ý từng mức rồi mới xem đáp án ở cuối.


**Lộ trình đầy đủ:** đề → nền tảng → baseline → điểm mấu chốt → code → trace → chứng minh → nộp LeetCode → Big-O → lỗi → biến thể → 6 bài tập → gợi ý → đáp án → checklist.

## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Cho `prices[i]` là giá cổ phiếu ngày i. Được mua **một lần**, bán **một lần vào ngày sau**, hoặc không giao dịch. Trả lợi nhuận **lớn nhất không âm**.

### Ví dụ quan trọng

`[7,1,5,3,6,4] → 5` (mua 1, bán 6); `[7,6,4,3,1] → 0` (không có giao dịch lời).

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [ ]:
from __future__ import annotations
from typing import List, Optional
from collections import deque, Counter

### 2.1. `list` là gì? Thay đổi từng phần tử bằng chỉ số

Trong Python, `list` là dãy có thứ tự. Ví dụ `[10,20,30]` chứa ba phần tử ở chỉ số
`0`, `1`, `2`. Truy cập `a[i]` để đọc, gán `a[i] = value` để sửa phần tử tại vị trí i.
List là **mutable** (có thể thay đổi sau khi tạo). Vì vậy thuật toán tối ưu bộ nhớ có thể
tái sử dụng list hiện tại; nhưng sửa list quá sớm cũng có thể phá dữ liệu còn cần dùng.

`len(a)` trả số phần tử, còn chỉ số cuối là `len(a)-1`. Đọc `a[len(a)]` gây `IndexError`.
Hãy dự đoán giá trị cuối của list trước khi chạy thí nghiệm.

In [ ]:
list_demo=[10,20,30]
print('ban đầu:',list_demo,'độ dài:',len(list_demo))
print('vị trí 0:',list_demo[0],'vị trí cuối:',list_demo[-1])
list_demo[1]=99
print('sửa phần tử thứ hai:',list_demo)
assert list_demo==[10,99,30]

### 2.2. `for`, `range` và vì sao ranh giới phải chính xác

`for i in range(n)` duyệt các số nguyên `0,1,...,n-1`: *n không nằm trong range*.
`range(start, stop, step)` điều chỉnh điểm bắt đầu, điểm dừng và bước nhảy.
Nếu `step` âm, vòng lặp đi từ phải qua trái; đây là công cụ thiết yếu cho cập nhật
*in-place* (đặc biệt quan trọng ở Pascal II).

Luôn viết thử 3–4 giá trị chỉ số bằng tay trước khi thêm `a[i+1]` hoặc `a[i-1]`.
Việc chọn sai biên có thể gây truy cập ngoài mảng hoặc bỏ sót phần tử cuối.

In [ ]:
print('range(4):', list(range(4)))
print('range(1,4):', list(range(1,4)))
print('range(3,0,-1):', list(range(3,0,-1)))
a_indices=[5,8,13,21]
for i in range(1,len(a_indices)):
    print('cặp liên tiếp:',a_indices[i-1],a_indices[i])

### 2.3. `append`, phép cộng list và sao chép an toàn

`a.append(x)` thêm phần tử `x` vào **chính list a** (không tạo list mới, trả về `None`).
`a + b` tạo một list mới nối các phần tử, không sửa hai list ban đầu.
`b = a` chỉ tạo **bí danh** của a; muốn sao chép list phẳng dùng `a.copy()` hoặc `a[:]`.

Bẫy thường gặp: nếu lưu các hàng Pascal bằng cùng một list rồi sửa, các hàng cũ cũng đổi theo.
Với mảng giá cổ phiếu, hãy phân biệt rõ *thay đổi dữ liệu* với *đọc giá trong ngày*.
Thí nghiệm dưới đây sẽ chỉ ra sai lầm do aliasing một cách trực quan.

In [ ]:
original_list=[1,2,1]
alias_list=original_list
copy_list=original_list.copy()
original_list.append(1)
print('original:',original_list,'alias:',alias_list,'copy:',copy_list)
print('append trả:',copy_list.append(2))
print('copy đã thay đổi:',copy_list)
assert len(alias_list)==4 and len(copy_list)==4

### 2.4. `enumerate` cho cả vị trí và giá trị

Nếu cần biết phần tử đang ở **vị trí** nào (ngày thứ mấy, hàng thứ mấy),
`enumerate(items)` tạo lần lượt cặp `(index, value)`.
Nếu chỉ cần giá trị, `for value in items` gọn hơn.
`list(enumerate(...))` giúp quan sát trực tiếp các cặp chỉ số/giá trị.

Lưu ý `0-indexed`: ngày đầu hoặc hàng đầu có số thứ tự **0** trong Python.
Đây là nguyên nhân rất thường gặp của lỗi *off-by-one*: lệch đúng một vị trí.

In [ ]:
example_values=[7,1,5,3]
print('các cặp:',list(enumerate(example_values)))
for pos,value in enumerate(example_values):
    print(f'vị trí {pos} giữ giá trị {value}')
assert list(enumerate([8]))==[(0,8)]

### 2.5. Gọi hàm `min/max`, không nhầm với chuỗi phép so sánh

`min(a,b)` trả giá trị nhỏ hơn; `max(a,b)` trả giá trị lớn hơn.
Biểu thức `best=max(best,candidate)` là quy tắc phổ biến khi lưu **đáp án tốt nhất đến thời điểm hiện tại**.
Bạn có thể viết lại tương đương thành `if candidate > best: best = candidate`.

Hãy chú ý **best** giữ thông tin lịch sử, không nhất thiết bằng giá trị lớn nhất của chính input.
Ở bài Pascal, `max` không dùng để chọn số từ hàng cũ; số giữa được tính **bằng tổng**.
Cùng cú pháp Python, nhưng ý nghĩa toán học quyết định thuật toán.

In [ ]:
best_demo=0
for candidate_demo in (2,-3,5,4):
    old_best=best_demo
    best_demo=max(best_demo,candidate_demo)
    print('thử',candidate_demo,':',old_best,'->',best_demo)
assert best_demo==5

### 2.6. Định nghĩa Big-O bằng một ví dụ số nhỏ

**Time Complexity** nói số thao tác tăng thế nào khi input dài hơn.
Duyệt một vòng qua n phần tử thường tốn `O(n)`; hai vòng lồng nhau mỗi vòng chạy n lần
có thể tốn `O(n²)`. Tuy vậy phải đếm **số lần chạy thực** chứ không chỉ nhìn số vòng for:
nếu vòng trong chạy độ dài 1,2,3,...,n thì tổng là `n(n+1)/2 = O(n²)`.

**Space Complexity** tính dữ liệu phụ; nếu đầu ra yêu cầu chứa n phần tử thì riêng output đã cần `O(n)`.
Trong Pascal I trả toàn bộ tam giác cần `O(n²)` cho chính output, Pascal II chỉ trả một hàng `O(n)`.
Mục tiêu học là biết *n* đại diện cho cái gì trong từng bài.

In [ ]:
for n in (3,5,10):
    triangle_cells=n*(n+1)//2
    print(f'n={n}: số ô của n hàng đầu = {triangle_cells}; n² = {n*n}')
assert sum(range(1,6))==15

### 2.7. Quan sát trạng thái trước và sau khi cập nhật

Muốn hiểu một thuật toán theo vòng lặp, cần trả lời ba câu:
(1) Trước vòng lặp ta đã biết gì? (2) Ở bước này thay đổi biến nào?
(3) Sau bước này thông tin nào tiếp tục đúng?
Ta gọi thông tin được giữ đúng ấy là **invariant** (bất biến vòng lặp).

Để tránh bị rối, in `old_state` **trước** lệnh sửa và `new_state` **sau** lệnh sửa.
Nếu in mỗi đáp án cuối, ta không tìm được nguyên nhân lỗi ở giữa vòng lặp.
Bài minh họa dưới chỉ kiểm tra thao tác giữ tổng chạy, chưa sử dụng thuật toán của bài chính.

In [ ]:
running_sum=0
for number in [2,4,-1]:
    before=running_sum
    running_sum += number
    print(f'đọc {number:>2}: trước={before:>2}; sau={running_sum:>2}')
assert running_sum==5

## 3. Python nền tảng — công cụ cần hiểu trước khi giải

`enumerate(prices)` tạo `(ngày,giá)`. `min_price` giữ **giá nhỏ nhất đã thấy trong quá khứ**; `profit = price - min_price` chỉ hợp lệ khi mua trước hoặc cùng ngày (cùng ngày lãi 0).

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [ ]:
prices=[7,1,5,3,6,4]
minimum=float('inf')
for day,price in enumerate(prices[:4]):
    minimum=min(minimum,price)
    print('ngày',day,'giá',price,'giá nhỏ nhất đến nay',minimum)

### 3.1. Trước khi tối ưu: dữ liệu là một dòng thời gian

Mảng `prices` biểu diễn giá cổ phiếu theo ngày: `prices[0]` là ngày đầu,
`prices[1]` là ngày tiếp, v.v. Bài chỉ cho phép **một lần mua và một lần bán**,
trong đó ngày mua phải **trước** ngày bán.

Vì thế cặp bất kỳ `(min(prices),max(prices))` không đủ:
gốc thời gian quyết định cặp mua/bán có hợp lệ hay không.
Ví dụ `[9,2,5]`: giá cao nhất 9 ở ngày 0 trước giá thấp nhất 2 ở ngày 1.
Không được “bán 9 trước rồi mua 2 sau”. Lợi nhuận hợp lệ tốt nhất là mua giá 2, bán giá 5, lời 3.

Khi không có giao dịch có lời, đáp án **0**, không phải số âm.

In [ ]:
prices_timeline121=[9,2,5]
for day,price in enumerate(prices_timeline121):
    print(f'ngày {day}: giá {price}')
print('max-min không hợp lệ:',max(prices_timeline121)-min(prices_timeline121))
print('lãi thực sự:',baseline(prices_timeline121) if 'baseline' in globals() else 3)
assert max(prices_timeline121)-min(prices_timeline121)==7

### Tự kiểm tra ý nghĩa từng dòng

- Đầu vào của mỗi biểu thức là gì?
- Dòng nào tạo cấu trúc mới và dòng nào thay đổi cấu trúc hiện có?
- `True`/`False` và `None` có nghĩa gì trong trường hợp này?
- Nếu dữ liệu rỗng hoặc chỉ có một phần tử, đoạn mã có còn chạy được không?

## 4. Cách nghĩ trực giác (baseline)

Brute Force thử mọi cặp ngày `(buy,sell)` với `buy < sell`, tính `prices[sell]-prices[buy]` rồi giữ tối đa. O(n²), trực quan vì đúng thứ tự ngày.

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [ ]:
def baseline(prices):
    best=0
    for buy in range(len(prices)):
        for sell in range(buy+1,len(prices)):
            best=max(best,prices[sell]-prices[buy])
    return best

In [ ]:
print(baseline([7,1,5,3,6,4]),baseline([7,6,4,3,1]))

### 4.1. Brute force thử mọi cặp ngày mua trước ngày bán

Cách rõ ràng nhất: hai vòng lặp tạo mọi cặp chỉ số `buy < sell`.
Với `n=4`, các cặp là `(0,1)`, `(0,2)`, `(0,3)`, `(1,2)`, `(1,3)`, `(2,3)`:
có đúng `n(n-1)/2=6` cặp.
Mỗi cặp tính `prices[sell]-prices[buy]` và cập nhật `best` nếu lớn hơn.
Khởi tạo best=0 vì luôn được phép **không giao dịch**.

Duyệt hai vòng không sai; nó giúp định nghĩa “đáp án tối ưu” một cách chắc chắn.
Nút thắt là tính lại rất nhiều khả năng mua khác nhau cho từng ngày bán.
Ta sẽ tối ưu chính điểm đó, không thay đổi tập giao dịch hợp lệ.

In [ ]:
pair_demo121=[7,1,5,3]
for buy in range(len(pair_demo121)):
    for sell in range(buy+1,len(pair_demo121)):
        profit=pair_demo121[sell]-pair_demo121[buy]
        print(f'mua ngày {buy} giá {pair_demo121[buy]}, bán ngày {sell} giá {pair_demo121[sell]} -> lãi {profit}')
assert baseline(pair_demo121)==4

### 4.2. Cặp giá cao nhất và thấp nhất không đủ xác nhận giao dịch

Xem `[9,8,7,6,1]`: chênh lệch max-min là 8,
nhưng giá 9 chỉ có ở ngày đầu và giá 1 chỉ có ở ngày cuối;
không thể mua giá 1 rồi bán giá 9 trong tương lai đã qua.
Mọi cặp hợp lệ đều lỗ, nên trả 0.

Xem `[1,5,2,9]`: min=1 xảy ra trước max=9,
chênh lệch 8 đúng là lợi nhuận tốt nhất.
Một số test tình cờ qua khi dùng max-min, nhưng thuật toán đó không đáp ứng ràng buộc thời gian.
**Dấu hiệu nhận diện:** đầu vào có thứ tự ngày/giờ và đề đòi “trước → sau” thì không thể bỏ chỉ số.

In [ ]:
for sequence in ([9,8,7,6,1],[1,5,2,9],[9,2,5]):
    naive_gap=max(sequence)-min(sequence)
    actual=baseline(sequence)
    print(sequence,'max-min=',naive_gap,'đáp án hợp lệ=',actual)
assert baseline([9,8,7,6,1])==0
assert baseline([1,5,2,9])==8

## 5. Vì sao cần chuyển sang kỹ thuật chính?

Ngày bán đang xét là hôm nay thì giá mua tốt nhất là **giá nhỏ nhất của các ngày trước đó**. Ta không cần thử lại tất cả ngày mua: chỉ giữ `min_price`.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

## 6. Lời giải chính — hiểu từng bước rồi mới viết

Duyệt từ trái qua phải: với mỗi giá, tính lợi nhuận nếu bán hôm nay rồi cập nhật lợi nhuận tốt nhất; sau đó cập nhật giá nhỏ nhất từng thấy. Bắt đầu `best=0` để cho phép không giao dịch.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

### 5.1. Bước đột phá: với mỗi ngày bán, chỉ nhớ giá mua thấp nhất trước đó

Giữ cố định một ngày bán j. Trong tất cả ngày mua i<j, lợi nhuận là
`prices[j]-prices[i]`. Để biểu thức lớn nhất, phải chọn `prices[i]` **nhỏ nhất**.
Vì vậy không cần thử mọi ngày mua; chỉ cần nhớ **minimum prefix** (giá thấp nhất đã thấy trong quá khứ).

Ta quét j từ trái sang phải và giữ hai trạng thái:
- `min_price`: giá mua nhỏ nhất trong các ngày đã xử lý;
- `best`: lợi nhuận tối đa trong tất cả giao dịch hợp lệ đã xét.

Với giá hôm nay p, xét `p-min_price` trước; sau đó cập nhật `min_price=min(min_price,p)`.
Kết quả đủ tốt để nộp: **O(n) thời gian, O(1) bộ nhớ phụ**. Đây là một dạng Greedy có bất biến rõ ràng.

In [ ]:
prefix_min121=float('inf')
for price in [7,1,5,3,6,4]:
    print('ngày có giá',price,': min quá khứ=',prefix_min121)
    prefix_min121=min(prefix_min121,price)
    print('  min sau khi xử lý=',prefix_min121)
assert prefix_min121==1

In [ ]:
def solve(prices: list[int]) -> int:
    min_price=float('inf')
    best=0
    for price in prices:
        best=max(best,price-min_price)
        min_price=min(min_price,price)
    return best

### 6.1. Giải thích ý nghĩa `float("inf")` và `best=0`

`float('inf')` là số vô cùng dương của Python: mọi giá cổ phiếu thông thường đều nhỏ hơn nó.
Bởi vậy `min_price=min(inf,price_first)` sẽ trở thành giá ngày đầu.
Ở vòng đầu, `price_first-inf = -inf`; so với `best=0` thì best vẫn 0.

Nếu muốn tránh vô cùng, có thể khởi tạo `min_price=prices[0]` và lặp từ ngày 1,
nhưng phải biết input không rỗng. Theo constraints của #121, `prices` có ít nhất một phần tử.
Cách dùng `inf` tạo code gọn và còn làm việc với list rỗng (một mở rộng không bắt buộc).
Khi học, quan trọng là **tại sao** đặt như vậy, không phải thuộc lòng cú pháp.

In [ ]:
from math import inf
print('inf lớn hơn 100?',inf>100)
print('giá đầu - inf:',7-inf)
print('best có đổi?',max(0,7-inf))
assert min(inf,7)==7
assert max(0,7-inf)==0

### 6.2. Cập nhật `best` rồi `min_price` có ý nghĩa thời gian ra sao?

Nếu đang xét giá p của ngày j, `min_price` trước cập nhật chỉ chứa thông tin
của những ngày **đã đi qua** (các ngày trước j).
`candidate=p-min_price` là giao dịch mua trước và bán hôm nay.
Sau đó `min_price=min(min_price,p)` để ngày j trở thành ứng viên mua cho những ngày **tương lai**.

Nếu cập nhật min trước thì lợi nhuận tối ưu **số học** của bài #121 vẫn không đổi,
vì mua/bán cùng ngày chỉ cho lãi 0 và best khởi tạo 0.
Nhưng cách “tính lãi trước, lưu giá hôm nay sau” bám sát yêu cầu *mua trước bán sau*,
tránh rối khi làm biến thể trả chỉ số ngày, tính phí, thời gian chờ, v.v.

In [ ]:
def compare_update_orders121(prices):
    before=float('inf');a=0
    for p in prices:
        a=max(a,p-before)
        before=min(before,p)
    after=float('inf');b=0
    for p in prices:
        after=min(after,p)
        b=max(b,p-after)
    return a,b
for vals in ([7,1,5,3,6,4],[9,8,7],[1,1,1]):
    left,right=compare_update_orders121(vals)
    print(vals,'cả hai thứ tự cho giá trị',left,right)
    assert left==right

## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| Ngày/giá | min trước | lời nếu bán hôm nay | best sau | min sau |
|---|---:|---:|---:|---:|
| 0 / 7 | ∞ | -∞ | 0 | 7 |
| 1 / 1 | 7 | -6 | 0 | 1 |
| 2 / 5 | 1 | 4 | 4 | 1 |
| 4 / 6 | 1 | **5** | **5** | 1 |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

### 7.1. Trace đầy đủ cả sáu ngày của ví dụ chuẩn

Xét `[7,1,5,3,6,4]`. Lưu `min_before` trước khi xử lý ngày j.

| j | Giá p | min trước | Lãi nếu bán hôm nay | best sau | min sau |
|---:|---:|---:|---:|---:|---:|
| 0 | 7 | ∞ | −∞ | 0 | 7 |
| 1 | 1 | 7 | −6 | 0 | 1 |
| 2 | 5 | 1 | 4 | 4 | 1 |
| 3 | 3 | 1 | 2 | 4 | 1 |
| 4 | 6 | 1 | **5** | **5** | 1 |
| 5 | 4 | 1 | 3 | 5 | 1 |

Mua ở ngày 1 giá 1, bán ở ngày 4 giá 6, lợi nhuận 5.
Không có cặp nào tốt hơn vì khi bán ngày j, ta đã xét giá mua thấp nhất trước đó.

In [ ]:
def trace_table121(prices):
    best=0;minimum=float('inf')
    print('ngày | p | min trước | thử bán | best sau | min sau')
    for j,p in enumerate(prices):
        before=minimum
        candidate=p-before
        best=max(best,candidate)
        minimum=min(minimum,p)
        print(j,p,before,candidate,best,minimum,sep=' | ')
    return best
assert trace_table121([7,1,5,3,6,4])==5

### 7.2. Trace trường hợp giảm dần: tại sao phải trả 0?

Với `[7,6,4,3,1]`, giá càng về sau càng nhỏ.
Ở mọi ngày bán j>0, giá p hôm nay **không cao hơn** giá mua thấp nhất trong quá khứ,
nên mọi `candidate` đều âm hoặc bằng 0.
Vì `best` bắt đầu bằng 0 và chỉ cập nhật bằng `max`, best không bao giờ giảm xuống âm.

Nếu một lời giải trả số âm cho input này, bạn đã ép thuật toán phải mua/bán
trong khi đề **cho phép không giao dịch**.
Nếu dùng hai biến `buy` và `sell`, hãy phân biệt điều kiện giao dịch hợp lệ với lợi nhuận dương.

In [ ]:
decreasing121=[7,6,4,3,1]
print('trace dãy giảm:')
assert trace_table121(decreasing121)==0
assert solve(decreasing121)==0

In [ ]:
def trace_stock(prices):
    minimum=float('inf');best=0
    for day,p in enumerate(prices):
        profit=p-minimum
        print(f'ngày={day}, giá={p}, min trước={minimum}, lời thử={profit}, best trước={best}')
        best=max(best,profit);minimum=min(minimum,p)
        print('  => min sau=',minimum,'best sau=',best)
    return best
trace_stock([7,1,5,3,6,4])

### 7.3. Thuật toán tương đương: lợi nhuận tốt nhất khi bán đúng hôm nay

Một cách diễn giải khác dùng mảng phụ `best_if_sell_today`:
đối với ngày j, phần tử đó là lợi nhuận tốt nhất khi bán vào **chính ngày j**,
được tính bằng `prices[j] - min(prices[:j])` cho j≥1.
Sau đó lấy `max(0,max(best_if_sell_today))`.

Cách dùng `min(prices[:j])` trực tiếp trong vòng lặp khiến ta tính lại các prefix,
tốn O(n²) thời gian tệ nhất. Tối ưu hóa là giữ một biến `min_price` cập nhật dần,
chuyển thuật toán thành O(n).
Thí nghiệm minh họa này giúp thấy **công việc lặp thừa** trước khi tối ưu.

In [ ]:
def all_sell_day_profits121(prices):
    return [prices[j]-min(prices[:j]) for j in range(1,len(prices))]
for values in ([7,1,5,3,6,4],[2,4,1],[7,6,4]):
    gains=all_sell_day_profits121(values)
    print(values,'lãi tốt nhất nếu bán ở từng ngày sau ngày đầu:',gains)
    assert max([0]+gains)==solve(values)

### 7.4. Một góc nhìn khác: tổng các thay đổi giá giữa ngày liên tiếp

Đặt `changes[j]=prices[j+1]-prices[j]`.
Mua ở ngày i và bán ở ngày j>i tương đương lấy tổng các thay đổi giá liên tiếp
`changes[i]+...+changes[j-1]` (các số ở giữa triệt tiêu).
Do đó bài #121 còn tương đương tìm **tổng đoạn con lớn nhất** của mảng `changes`,
cho phép chọn không giao dịch nên kết quả không âm.

Đây là cầu nối với thuật toán Kadane và bài Maximum Subarray.
Tuy nhiên với người mới, lưu `min_price` thường dễ suy luận hơn và dùng O(1) bộ nhớ.
Đoạn code minh họa dùng `changes` để kiểm chứng mối liên hệ, không thay thế lời giải chính.

In [ ]:
def kadane_from_changes121(prices):
    best=0;current=0
    for prev,next_price in zip(prices,prices[1:]):
        change=next_price-prev
        current=max(0,current+change)
        best=max(best,current)
    return best
for vals in ([7,1,5,3,6,4],[7,6,4,3,1],[1,2,3,4],[2,4,1]):
    print('giá',vals,'-> đáp án Kadane',kadane_from_changes121(vals))
    assert kadane_from_changes121(vals)==solve(vals)

## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Bất biến sau ngày i: `min_price` là giá nhỏ nhất các ngày đến i và `best` là lợi nhuận tốt nhất trong những cặp mua/bán đã xét. Khi đến ngày mới, bán ngày đó tốt nhất bằng `price - min_price` từ quá khứ. So với `best` cũ giữ đúng tối ưu toàn cục. Không có cặp lời thì 0.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [ ]:
class Solution:
    def maxProfit(self, prices: list[int]):
        min_price=float('inf')
        best=0
        for price in prices:
            best=max(best,price-min_price)
            min_price=min(min_price,price)
        return best

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [ ]:
for arr,expected in [([7,1,5,3,6,4],5),([7,6,4,3,1],0),([1],0),([2,4,1],2),([3,2,6,5,0,3],4)]:
    assert solve(arr)==expected
    assert Solution().maxProfit(arr)==expected
    assert baseline(arr)==expected
print('PASS 5 ca stock')

### 10.1. Liệt kê hết dãy giá ngắn để đối chiếu Brute Force với Greedy

Không chỉ test vài input ngẫu nhiên, ta có thể **liệt kê toàn bộ** dãy ngắn
mà mỗi giá thuộc tập `{1,2,3,4}`; với độ dài 1..5 chỉ có
`4+16+64+256+1024=1364` dãy.

Đối chiếu baseline O(n²), hàm solve O(n), code submit và phiên bản Kadane.
Tại đây brute force đóng vai trò nguồn kiểm tra độc lập, không phải lời giải để nộp.
Nếu có bất kỳ bất đồng, in dãy ngắn gây lỗi: đó thường là phản ví dụ dễ hiểu nhất.
Phép kiểm tra toàn bộ input nhỏ là công cụ đặc biệt hữu ích khi giải bài tối ưu một lần duyệt.

In [ ]:
from itertools import product
checked121=0
for n in range(1,6):
    for seq in product((1,2,3,4),repeat=n):
        prices=list(seq)
        truth=baseline(prices)
        assert solve(prices)==truth==Solution().maxProfit(prices)==kadane_from_changes121(prices)
        checked121+=1
print('PASS',checked121,'dãy giá ngắn liệt kê đầy đủ')
assert checked121==1364

### 10.2. Kiểm thử các thuộc tính biến đổi: cộng một hằng số không đổi lợi nhuận

Nếu mọi giá tăng thêm cùng một hằng số c, chênh lệch bán−mua không đổi:
`(sell+c)-(buy+c)=sell-buy`.
Do đó `maxProfit([p+c for p in prices])` phải bằng maxProfit(prices).
Tương tự, khi nhân tất cả giá với số dương k, lợi nhuận tối ưu nhân k
(trên số nguyên, không lo làm tròn).

Đây là **metamorphic testing**: kiểm tra mối quan hệ giữa hai input liên quan
mà không cần biết trước số đáp án cho mỗi dãy.
Ta thử nhiều dãy giá sinh ngẫu nhiên bằng seed cố định; tính chất này
bổ sung cho test mẫu mà không thay cho chứng minh tính đúng.

In [ ]:
import random
rng121=random.Random(121)
for _ in range(220):
    prices=[rng121.randint(1,100) for _ in range(rng121.randint(1,30))]
    expected=solve(prices)
    c=rng121.randint(-20,20)
    k=rng121.randint(1,5)
    assert solve([p+c for p in prices])==expected
    assert solve([p*k for p in prices])==expected*k
print('PASS 220 dãy với hai tính chất biến đổi')

## 11. Phân tích Time/Space Complexity

`n` ngày: **Time O(n)** (duyệt một lần), **Space O(1)** (hai biến). Baseline O(n²) time, O(1) space.

Nhớ phân biệt **số phần tử đầu vào** với **độ cao cây**, **số bit** hoặc **số chữ số**. Nếu thuật toán dùng recursion, bộ nhớ ngăn xếp lời gọi cũng cần được tính.

## 12. Các lỗi người mới hay mắc

- ❌ Lấy `max(prices)-min(prices)` không kiểm soát **thứ tự ngày**.
- ❌ Khởi tạo best là số âm → trường hợp giá giảm phải trả 0.
- ❌ Lưu min_price chưa rõ là của ngày nào; nên theo dõi trạng thái.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

### 12.1. Code sai: `return max(prices)-min(prices)`

Lỗi này bỏ qua chiều thời gian. Dãy `[9,1]` cho chênh lệch 8,
nhưng ngày bán duy nhất sau ngày mua có giá 1, mua 9 bán 1 sẽ lỗ 8;
không giao dịch là tốt hơn nên đáp án đúng 0.

Đừng sửa bằng cách “đảo ngược list”: bài toán vẫn cần chọn **mua trước bán sau**,
đảo list sẽ thay đổi ý nghĩa từng ngày.
Cần lưu **minimum prefix** theo tiến trình quét từ trái sang phải,
không dùng cực trị trên toàn bộ dãy như thể dữ liệu là tập hợp không có thứ tự.

In [ ]:
for input_prices in ([9,1],[7,6,4,3,1],[1,9]):
    wrong=max(input_prices)-min(input_prices)
    correct=solve(input_prices)
    print('giá:',input_prices,'max-min:',wrong,'đúng:',correct)
assert solve([9,1])==0
assert max([9,1])-min([9,1])==8

### 12.2. Code sai: khởi tạo lợi nhuận tốt nhất bằng số âm

Giả sử viết `best=prices[1]-prices[0]` và không so với 0.
Khi mọi cặp đều lỗ, best có thể là số âm — trong khi bài cho phép **không giao dịch**.
Thật ra có hai lựa chọn tương đương nếu muốn ép kiểu giao dịch:
`max(0,best_gain)` ở cuối hoặc giữ `best=0` ngay từ đầu.
Cách thứ hai rõ ràng hơn: bất biến best luôn là lợi nhuận lớn nhất **không âm**.

Một lỗi nữa: dùng `float('inf')` cho `best` thay vì `min_price`,
khiến `max(best,candidate)` không bao giờ đổi. Hãy đặt tên biến theo ý nghĩa toán học.

In [ ]:
def wrong_must_trade121(prices):
    return max(prices[j]-prices[i]
               for i in range(len(prices)) for j in range(i+1,len(prices)))
example_down121=[4,3,2,1]
print('bắt buộc giao dịch (không đúng đề):',wrong_must_trade121(example_down121))
print('đúng #121:',solve(example_down121))
assert wrong_must_trade121(example_down121)==-1
assert solve(example_down121)==0

## 13. Nhận diện dạng bài và vận dụng

Đề nói "mua trước, bán sau", chỉ một giao dịch, tối ưu hiệu giữa hai phần tử **có thứ tự** → tiền tố nhỏ nhất / greedy one-pass.

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Thử một dãy luôn giảm và một dãy tăng hoàn toàn: giá cao nhất có thể đến trước giá thấp nhất nên không được mua-bán ngược thời gian.

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [ ]:
for a in [[9,8,7,6],[1,2,3,4],[2,4,1]]: print(a,'->',solve(a))

### 14.1. Muốn biết ngày mua/bán cụ thể: phải giữ cả chỉ số

Bài #121 chỉ trả **số tiền lời**, nhưng biến thể có thể yêu cầu cặp `(buy_day,sell_day)`.
Khi cập nhật `min_price`, hãy đồng thời ghi `min_day`.
Khi tìm được `candidate>best`, lưu cặp `(min_day, current_day)`.
Nếu có nhiều cặp cùng mức lời, dấu `>` (không phải `>=`) giữ cặp tối ưu đầu tiên đã tìm thấy.

Điểm tinh tế: cập nhật `best` **trước** khi cập nhật `min_price` để cặp giao dịch
luôn có ngày mua cũ hơn ngày bán. Bộ nhớ vẫn O(1), thời gian O(n).
Đây là biến thể thực tế giúp luyện kỹ năng giữ **witness** (nhân chứng) của đáp án.

In [ ]:
for seq in ([7,1,5,3,6,4],[7,6,4,3,1],[2,4,1,4]):
    trade=answer_best_trade_indices(seq) if 'answer_best_trade_indices' in globals() else None
    print('giá:',seq,'đáp án lãi:',solve(seq),'cặp ngày: (xem phần Đáp án)',trade)
assert solve([2,4,1,4])==3

### 14.2. Vì sao không được áp dụng y hệt cho bài mua bán nhiều lần?

LeetCode #121 chỉ cho phép **một giao dịch**. Nếu đề đổi thành mua/bán nhiều lần,
quy tắc thay đổi: có thể tận dụng nhiều đoạn tăng độc lập, tổng lời lớn hơn một giao dịch.
Ví dụ `[1,5,2,9]`: một giao dịch tốt nhất mua 1 bán 9 lời 8;
nhưng nếu được mua/bán nhiều lần, có thể lời `(5−1)+(9−2)=11`.

Pattern “minimum prefix” giải đúng **một cặp mua/bán**;
nó không tự động giải bài #122. Hãy đọc kỹ constraints “at most one transaction”
trước khi chọn thuật toán. Đây là cách nhận diện biến thể quan trọng để tránh áp dụng nhầm kỹ thuật.

In [ ]:
def multi_transactions121(prices):
    total=0
    for day in range(1,len(prices)):
        total+=max(0,prices[day]-prices[day-1])
    return total
sample_multi121=[1,5,2,9]
print('một giao dịch:',solve(sample_multi121))
print('nhiều giao dịch:',multi_transactions121(sample_multi121))
assert solve(sample_multi121)==8
assert multi_transactions121(sample_multi121)==11

## 15. Tự luyện — 5 bài tăng dần

**Mục tiêu:** tự hiểu cú pháp, trace bằng tay, viết lại thuật toán, chuyển sang biến thể, rồi thực hiện mini challenge. Tất cả ví dụ giữ nguyên đúng bản chất đề LeetCode. **Đừng xem đáp án trước.**

### Bài 1 — Dự đoán output của Python (dễ)

Cho `prices=[7,1,5,3]`, đoán `list(enumerate(prices))` và kết quả của `min(prices[:2])`.

Hãy viết output ra giấy trước khi chạy cell dưới.

In [ ]:
prices=[7,1,5,3]
print(list(enumerate(prices)))
print(min(prices[:2]))

### Bài 2 — Trace bằng tay (hiểu bản chất)

Lập bảng cho `[7,1,5,3,6,4]`: ở mỗi ngày ghi giá thấp nhất *trước khi xử lý ngày đó*, lời thử nếu bán hôm nay, và lợi nhuận tốt nhất đến nay.

Tự viết bảng gồm **trạng thái trước**, phép kiểm tra và **trạng thái sau**; không chỉ ghi kết quả cuối.

### Bài 3 — TODO: Tự viết lời giải chính

Viết `exercise_max_profit(prices)` giải trong O(n) thời gian và O(1) bộ nhớ. Chỉ được mua **trước khi** bán.

Các hàm bài tập được định nghĩa độc lập và **không được gọi** khi chưa hoàn thiện.

In [ ]:
def exercise_max_profit(prices):
    # TODO: min_price, best; cập nhật qua từng giá.
    raise NotImplementedError

### Bài 4 — TODO: Chuyển giao sang biến thể

Viết `exercise_best_trade_indices(prices)` trả `(buy_index,sell_index)` của giao dịch lãi nhất; nếu không có lãi trả `(-1,-1)`; khi hòa chọn cặp tìm thấy đầu tiên.

Hãy thử input nhỏ nhất và một phản ví dụ. Dự đoán output trước khi gọi.

In [ ]:
def exercise_best_trade_indices(prices):
    # TODO: lưu giá thấp nhất và chỉ số của nó; cập nhật khi lợi nhuận > best.
    raise NotImplementedError

### Bài 5 — TODO: Mini challenge

Mini challenge: `exercise_profitable_trades(prices)` liệt kê mọi giao dịch có lời, kết quả là các tuple `(ngày_mua, ngày_bán, lợi_nhuận)` theo thứ tự i rồi j. Đây là brute force O(n²) để so sánh với cách tối ưu.

Hãy nêu độ phức tạp của hàm và thử tối thiểu 3 ca biên.

In [ ]:
def exercise_profitable_trades(prices):
    # TODO: hai vòng i<j, chỉ append nếu prices[j] > prices[i].
    raise NotImplementedError

### Bài 6 — TODO: giao dịch phải cách nhau tối thiểu `gap` ngày

Đây là mini challenge **chuyển giao** thuật toán minimum prefix.
Viết `exercise_profit_min_gap(prices,gap)` với `gap>=1`:
chỉ được mua ngày i, bán ngày j nếu **`j-i >= gap`**; có tối đa một giao dịch;
không có giao dịch lời hoặc không đủ ngày thì trả 0.

Ví dụ `[1,9,3,10]`, gap=2: mua ngày 0 bán ngày 3 lời 9;
mua ngày 1 bán ngày 3 lời 1 **cũng hợp lệ** vì 3−1=2;
nhưng tối ưu vẫn là 9. Với `[1,9]`, gap=2 không có cặp hợp lệ, trả 0.

Gợi ý 1: khi đang bán ngày j, chỉ được xét ngày mua ≤ j−gap.
Gợi ý 2: mỗi lượt thêm giá `prices[j-gap]` vào tập các giá mua **đủ điều kiện**.
Gợi ý 3: giữ minimum giá đủ điều kiện và best, một lần quét O(n).

In [ ]:
def exercise_profit_min_gap(prices,gap):
    # TODO: quét ngày bán j từ gap đến n-1, dùng minimum của các ngày i <= j-gap.
    # Không viết hai vòng lặp lồng nhau nếu bạn muốn đạt O(n).
    raise NotImplementedError

# Sau khi tự làm:
# assert exercise_profit_min_gap([1,9],2)==0
# assert exercise_profit_min_gap([1,9,3,10],2)==9

## 16. Gợi ý mở dần — đừng đọc cùng lúc

<details><summary><b>Cấp 1 · Trực giác</b></summary>

Ngày mua phải trước ngày bán; đừng lấy giá `max` và `min` toàn bộ mà quên thứ tự.

</details>

<details><summary><b>Cấp 2 · Dữ liệu và cú pháp</b></summary>

Khi xét ngày bán hiện tại, giữ giá mua nhỏ nhất **đã đi qua** và lợi nhuận tốt nhất.

</details>

<details><summary><b>Cấp 3 · Gần-code</b></summary>

Bước 3: `best=max(best,price-min_price)` rồi `min_price=min(min_price,price)`; bước 4 nhớ cả chỉ số ngày mua.

</details>

## 17. Đáp án tham khảo — chỉ đọc sau khi tự làm

### Bài 1 — Output

Dòng 1: `[(0, 7), (1, 1), (2, 5), (3, 3)]`. Dòng 2: `1`. `enumerate` trả (chỉ số ngày, giá).

### Bài 2 — Trace tay

Ngày 0 không có giá mua cũ; sau ngày 0 min=7, best=0. Ngày 1 giá 1: lời thử=-6, min=1, best=0. Ngày 2 giá 5: lời thử=4, best=4. Ngày 3 giá 3: lời thử=2, best=4. Ngày 4 giá 6: lời thử=5, best=5. Ngày 5 giá 4: lời thử=3, best=5. **Đáp án 5**.

### Đáp án bài 3 — Viết lại giải pháp chính

Đọc rõ base case và ý nghĩa giá trị trả về. Đáp án độc lập, không dựa vào `solve` phía trên.

In [ ]:
def answer_max_profit(prices):
    min_price=float("inf");best=0
    for price in prices:
        best=max(best,price-min_price)
        min_price=min(min_price,price)
    return best

### Đáp án bài 4 — Biến thể

So sánh **trạng thái cần mang theo** giữa bài chính và biến thể.

In [ ]:
def answer_best_trade_indices(prices):
    min_price=float("inf");min_day=-1
    best=0;trade=(-1,-1)
    for day,price in enumerate(prices):
        profit=price-min_price
        if profit>best:
            best=profit;trade=(min_day,day)
        if price<min_price:
            min_price=price;min_day=day
    return trade

### Đáp án bài 5 — Mini challenge

Kiểm tra kỹ trường hợp rỗng, đầu/cuối, và các điều kiện dừng.

In [ ]:
def answer_profitable_trades(prices):
    ans=[]
    for i in range(len(prices)):
        for j in range(i+1,len(prices)):
            profit=prices[j]-prices[i]
            if profit>0: ans.append((i,j,profit))
    return ans

### Đáp án bài 6 — minimum prefix của các ngày mua **đã đủ thời gian chờ**

Với ngày bán j, những ngày được phép mua là `i<=j-gap`.
Do đó khi xử lý j, ta mới đưa giá `prices[j-gap]` vào tập giá mua hợp lệ
bằng cách cập nhật `minimum=min(minimum,prices[j-gap])`.
Sau đó xét lời thử `prices[j]-minimum`, cập nhật best.

Mỗi giá được đưa vào trạng thái một lần, mỗi ngày bán xét một lần:
Time O(n), Space O(1). Nhờ giới hạn hợp lệ thể hiện rõ trong index `j-gap`,
đáp án không dựa vào may mắn của thứ tự ngày.
Kết quả cũng đối chiếu được với brute force cặp `(i,j)` có `j-i>=gap`.

In [ ]:
def answer_profit_min_gap(prices,gap):
    minimum=float('inf');best=0
    for sell in range(gap,len(prices)):
        minimum=min(minimum,prices[sell-gap])
        best=max(best,prices[sell]-minimum)
    return best

def oracle_profit_min_gap(prices,gap):
    return max([0]+[prices[j]-prices[i]
                    for i in range(len(prices)) for j in range(i+gap,len(prices))])
for prices in ([1,9],[1,9,3,10],[7,1,5,3,6,4],[5,4,3,2]):
    for gap in (1,2,3,4,10):
        assert answer_profit_min_gap(prices,gap)==oracle_profit_min_gap(prices,gap)
print('PASS bài 6: minimum gap và brute force')

### Kiểm thử 3 lời giải bài tập

Các assert sau **chỉ kiểm tra đáp án mẫu**, không chạy các hàm TODO của bạn. Khi hoàn thiện TODO, bạn hãy tự thay tên hàm trong test để đánh giá.

In [ ]:
assert answer_max_profit([7,1,5,3,6,4])==5
assert answer_max_profit([7,6,4,3,1])==0
assert answer_best_trade_indices([7,1,5,3,6,4])==(1,4)
assert answer_best_trade_indices([7,6,4,3,1])==(-1,-1)
assert answer_profitable_trades([3,1,4])==[(0,2,1),(1,2,3)]
assert answer_profitable_trades([3,2,1])==[]
print("PASS: đáp án 3 bài coding của phần luyện tập")

## 18. Tóm tắt và checklist submit

**Nhớ nhanh:** Ngày bán đang xét là hôm nay thì giá mua tốt nhất là **giá nhỏ nhất của các ngày trước đó**. Ta không cần thử lại tất cả ngày mua: chỉ giữ `min_price`.

- [ ] Tôi tự nói được input, output và điều kiện biên.
- [ ] Tôi hiểu từng cú pháp Python trong thuật toán.
- [ ] Tôi có thể tự vẽ bảng trace ít nhất một ví dụ.
- [ ] Tôi giải thích được **tại sao đúng** mà không dựa vào test.
- [ ] Tôi phân biệt được Time/Space của baseline và lời giải chính.
- [ ] Tôi copy đúng phần class LeetCode, không copy hàm dựng cây/list hay `print()`.
- [ ] Tôi tự làm xong các TODO, rồi mới đọc đáp án.

**Bài tiếp theo:** duy trì thói quen trace từ trạng thái nhỏ và chỉ tối ưu khi đã hiểu bản chất.